In [2]:
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from neo4j import GraphDatabase

In [3]:
load_dotenv()

uri = os.getenv("NEO4J_URI")
user = os.getenv("NEO4J_USER")
password = os.getenv("NEO4J_PASSWORD")
database = os.getenv("NEO4J_DATABASE", "neo4j")

driver = GraphDatabase.driver(uri, auth=(user, password))

out_dir = Path("../data/processed/neo4j_import")

In [5]:
def load_table(filename):
  df = pd.read_csv(out_dir / filename)
  return df.astype(object).where(pd.notna(df), None)

artists = load_table("artists.csv")
release_groups = load_table("release_groups.csv")
releases = load_table("releases.csv")
recordings = load_table("recordings.csv")
persons = load_table("persons.csv")
tags = load_table("tags.csv")

created = load_table("created.csv")
has_release = load_table("has_release.csv")
has_track = load_table("has_track.csv")
member_of = load_table("member_of.csv")
has_tag = load_table("has_tag.csv")

In [ ]:
# with driver.session(database=database) as session:
#     session.run("MATCH (n) DETACH DELETE n")

In [7]:
def run_query(query, rows):
    with driver.session(database=database) as session:
        session.run(query, rows=rows)

run_query("""
UNWIND $rows AS row
MERGE (a:Artist {artist_id: row.artist_id})
SET a.name = row.name
""", artists.to_dict("records"))

run_query("""
UNWIND $rows AS row
MERGE (rg:ReleaseGroup {release_group_id: row.release_group_id})
SET rg.title = row.title
""", release_groups.to_dict("records"))

run_query("""
UNWIND $rows AS row
MERGE (r:Release {release_id: row.release_id})
SET r.title = row.title,
    r.release_date = row.release_date,
    r.country = row.country,
    r.medium_format = row.medium_format
""", releases.to_dict("records"))

run_query("""
UNWIND $rows AS row
MERGE (rec:Recording {recording_id: row.recording_id})
SET rec.title = row.title,
    rec.length_ms = row.length_ms,
    rec.length_min = row.length_min
""", recordings.to_dict("records"))

run_query("""
UNWIND $rows AS row
MERGE (p:Person {person_id: row.person_id})
SET p.name = row.name
""", persons.to_dict("records"))

run_query("""
UNWIND $rows AS row
MERGE (t:Tag {name: row.name})
""", tags.to_dict("records"))

In [8]:
run_query("""
UNWIND $rows AS row
MATCH (a:Artist {artist_id: row.artist_id})
MATCH (rg:ReleaseGroup {release_group_id: row.release_group_id})
MERGE (a)-[:CREATED]->(rg)
""", created.to_dict("records"))

run_query("""
UNWIND $rows AS row
MATCH (rg:ReleaseGroup {release_group_id: row.release_group_id})
MATCH (r:Release {release_id: row.release_id})
MERGE (rg)-[:HAS_RELEASE]->(r)
""", has_release.to_dict("records"))

run_query("""
UNWIND $rows AS row
MATCH (r:Release {release_id: row.release_id})
MATCH (rec:Recording {recording_id: row.recording_id})
MERGE (r)-[rel:HAS_TRACK]->(rec)
SET rel.position = row.position
""", has_track.to_dict("records"))

In [9]:
member_of["roles"] = member_of["roles"].apply(
    lambda x: [] if x is None or x == "" else str(x).split(";")
)

run_query("""
UNWIND $rows AS row
MATCH (p:Person {person_id: row.person_id})
MATCH (a:Artist {artist_id: row.artist_id})
CREATE (p)-[rel:MEMBER_OF]->(a)
SET rel.begin = row.begin,
    rel.end = row.end,
    rel.roles = row.roles
""", member_of.to_dict("records"))

In [10]:
run_query("""
UNWIND $rows AS row
MATCH (a:Artist {artist_id: row.artist_id})
MATCH (t:Tag {name: row.tag})
MERGE (a)-[rel:HAS_TAG]->(t)
SET rel.count = row.count
""", has_tag.to_dict("records"))

In [11]:
with driver.session(database=database) as session:
    result = session.run("""
    MATCH (n)
    RETURN labels(n)[0] AS label, count(n) AS count
    ORDER BY label
    """)
    
    for record in result:
        print(record["label"], record["count"])

Artist 6
Person 44
Recording 70
Release 6
ReleaseGroup 6
Tag 45


#### Neo4j Import Result

The Neo4j import succeeded. The database now contains 177 nodes and 209 relationships across the Graph Schema v0 design.

Node labels:
- Artist
- ReleaseGroup
- Release
- Recording
- Person
- Tag

Relationship types:
- CREATED
- HAS_RELEASE
- HAS_TRACK
- MEMBER_OF
- HAS_TAG

Sanity queries confirmed that:
- catalog traversal works, such as Nirvana -> Bleach -> tracklist
- shared-member relationships work, such as Matt Cameron and Jason Everman
- the graph contains both catalog structure and relationship-rich discovery structure